In [1]:
import numpy as np
import math

#### 1.sigmoid

公式如下：

$$ \sigma(x) = \frac{1}{1+e^{-x}} $$

$$ \sigma'(x) = \sigma(x)(1 - \sigma(x)) $$

缺点：

- 算出来梯度较小，易造成梯度消失；传播n层后：$ 0.25^{n}x $ 

$$ \sigma'(x) = \frac{e^{-x}}{(1 + e^{-x})^2} = \frac{1}{e^{-x} + 2 + e^{x}} \leq \frac{1}{4} $$

In [2]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

#### 2.ReLU

公式如下：

$$ f(x) = \begin{cases}
          x, & x > 0 \\
          0, & x \leq 0 
          \end{cases} $$

或者：

$$ f(x) = \max(0, x) $$

优点：
- 解决了梯度消失问题
- 计算效率高
- 稀疏激活性：截断梯度为0的那部分的神经元

缺点：
- 神经元死亡：某个神经元输出为0之后永远不会更新

$$ \frac{\partial L}{\partial f} · \frac{\partial f}{\partial x} = 0 $$

其中，$ f(x) = 0 $

In [3]:
def ReLU(x):
    # maximum 是逐个对比
    return np.maximum(0, x)

### 

### 3. Tanh

公式如下：

$$ Tanh(x) = \frac{e^{x} - e^{-x}}{e^{x} + e^{-x}} $$
$$ Tanh'(x) = 1 - Tanh^{2}(x) $$

缺点：

令 $ Tanh(x) = t $，$ e^{x} = m $，于是有

$$ (t - 1) x^{2} + t + 1 = 0 $$
$$ \Delta = 0 - 4(t - 1)(t + 1) \geq 0 $$

解得 $ 0 \leq t = Tanh(x) \leq 1 $，当且仅当 $ x = 0 $ 时取等。比之 $ sigmoid(x) \leq \frac{1}{4} $ 强一点，但还是会有梯度消失问题

In [4]:
def Tanh(x):
    return (np.exp(x) - np.exp(-x)) / (np.exp(x) + np.exp(-x))

### 4. SiLU
ReLU和sigmoid的孩子

公式如下：

$$ SiLU(x) = x \cdot \sigma(x) $$

将sigmoid作为一个门控函数接入，既基本保留ReLU正向线性的特性，又用sigmoid的平滑曲线替代了ReLU的硬截断

<span style="color: red">负值梯度不会像 ReLU 那样完全死掉</span>

门控函数：控制信息通过的比例

In [ ]:
def SiLU(x, beta=1.0):
    return x * sigmoid(x * beta)

### 5. GeLU

公式如下：

$$\mathrm{GELU}(x) = x \cdot \Phi(x)$$

推导：

$$ erf(x) = \int_{0}^{x} \frac{2}{\sqrt{\pi}}\, e^{u^2}\, du $$

$$ \Phi(x) = \int_{-\infty}^{0} \frac{1}{\sqrt{2\pi}}\, e^{-\frac{t^2}{2}}\, dt + \int_{0}^{x} \frac{1}{\sqrt{2\pi}}\, e^{-\frac{t^2}{2}}\, dt = \frac{1}{2} +  \int_{0}^{x} \frac{1}{\sqrt{2\pi}}\, e^{-\frac{t^2}{2}}\, dt $$

$$ = \int_{0}^{x} \frac{1}{\sqrt{2\pi}}\, e^{-\frac{t^2}{2}}\, dt =  \int_{0}^{\frac{x}{\sqrt{2}}} \frac{1}{\sqrt{\pi}}\, e^{-u^2}\, du = \frac{1}{2}erf(\frac{x}{\sqrt{2}})$$

所以有

$$ GeLU(x) = \frac{x}{2}(1 + erf(\frac{x}{\sqrt{2}}))$$

门控函数改为正态分布函数，负数信息量减少了一些

In [6]:
def GeLU(x):
    return (x / 2) * (1 + np.vectorize(math.erf)(x / np.sqrt(2)))

### 6. SwiGLU

英文全称：<span style="color: red">**Swi**</span>sh-<span style="color: red">**G**</span>ated<span style="color: red">**L**</span>inear<span style="color: red">**U**</span>nit

该激活函数的效果来自于神的恩赐？

公式如下：

$$ SwiGLU(x) = (W_1x) \cdot SiLU(W_2x) $$

优点：

- 可学习性:用SiLU函数当门控函数,网络能自主学习放行多少信息
- 灵活性:SiLU能输出负值、大于1的值,更平滑灵活。

实证(Shazeer 2020)表明,在 Transformer FFN 里 SwiGLU 困惑度最低,且参数不增加甚至更省——所以 LLaMA、Mistral 等大模型都用它

In [7]:
def SwiGLU(W1, W2, x):
    return (W1 @ x) * SiLU(W2 @ x)

### 7. softmax

高维形式的softmax

公式如下：

$$ x_k = \frac{e^{x_k}}{\sum\limits_{i=0}^{n} e^{x_i}} $$

In [8]:
def softmax(x):
    # 数值稳定
    x = x - np.max(x)

    return np.exp(x) / np.sum(np.exp(x), axis=-1, keepdims=True)